In [1]:
from hexgrid import HexCoord, HexGrid, build_obstacle_map, VelocityState, get_neighbors_at_radius, hdist
from hexstarrl import ReplayBuffer, ExperienceCollector, EnvironmentStore, HUNetTDTrainer, HUNetStateEncoder, ValueNetwork_HUNet, RLState

from hexstar import HStarProblem, HStarSearch, h_cost_travel_time_euclidean, construct_full_solution, BidHStarSearch, calculate_path_smoothness, travel_time_euclidean
from hexplot import plot_hex_grid_2

from tqdm.auto import tqdm

from dataclasses import dataclass

import random
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from collections import defaultdict
import torch.nn as nn


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)


Device: cuda


In [3]:
center = HexCoord(0,0)
start_v = VelocityState(0, None) # 0 speed, unspecified direction (allows expansion in any direction at root)
goal_v = VelocityState(0, None) # matches any direction as long as location matches. 

GOAL=HexCoord(q=-2, r=-2)


In [4]:
a_max = 3 # acceleration ability
a_min = 1 # deceleration ability
a_lat = 7 # maximum lateral g-force allowed by the agent

In [5]:
num_maps = 20
radius = 50
hex_size = 1
obs_densities = [0.05, 0.30] # Random range

In [6]:
n_step = 1

In [7]:
def random_start_goal(center, radius, min_distance_ratio=0.5):
    while True:
        q1 = random.randint(-radius + 1, radius - 1)
        r1 = random.randint(
            max(-radius + 1, -q1 - radius + 1),
            min(radius - 1, -q1 + radius - 1)
        )

        q2 = random.randint(-radius + 1, radius - 1)
        r2 = random.randint(
            max(-radius + 1, -q2 - radius + 1),
            min(radius - 1, -q2 + radius - 1)
        )

        start = HexCoord(center.q + q1, center.r + r1)
        goal = HexCoord(center.q + q2, center.r + r2)

        distance = (
            abs(start.q - goal.q)
            + abs(start.r - goal.r)
            + abs((start.q + start.r) - (goal.q + goal.r))
        ) // 2

        if distance >= radius * min_distance_ratio:
            return start, goal

In [8]:
replay_buffer = ReplayBuffer(
    capacity=2_000_000,
    solution_fraction=0.20,
)
environment_store = EnvironmentStore()
experience_collector = ExperienceCollector(replay_buffer, environment_store)

encoder = HUNetStateEncoder()

value_network = ValueNetwork_HUNet(
    encoder=encoder,
)

trainer = HUNetTDTrainer(
    model=value_network,
    replay_buffer=replay_buffer,
    environment_store=environment_store,
    learning_rate=1e-3,

    solution_weight=0.7,
    nonterminal_weight=0.3,
)

In [9]:
# Populate the environment store

map_configs = [
    (
        "clustered_n",
        {
            "n": 100,
            "min_cluster_size": 40,
            "max_cluster_size": 60,
            "exclude_radius": 5,
        },
    ),
    # (
    #     "random_n",
    #     {
    #         "n": 100,
    #     },
    # ),
]



from tqdm.auto import tqdm

progress = tqdm(
    range(num_maps),
    desc="Generating maps",
    unit="map",
)
for seed in progress:
    obs_density = random.uniform(*obs_densities)
    n = int(obs_density * 3 * radius * (radius + 1))
    
    map_type, map_params = random.choice(map_configs)
    #print(map_type)
    map_params['n'] = n
    start, goal = random_start_goal(
        center,
        radius,
        min_distance_ratio=0.60,
    )

    obstacles = set(
        build_obstacle_map(
            center=center,
            radius=radius,
            seed=seed,
            exclude=[start, goal],
            types=[
                (
                    map_type,
                    map_params.copy(),
                )
            ],
        ).keys()
    )

    obstacles.update(
        get_neighbors_at_radius(
            HexCoord(0, 0),
            radius,
        )
    )

    hex_grid = HexGrid(
        hex_size,
        obstacles = obstacles,
        center = center,
        radius = radius,
    )
    environment_store.add(
        seed, # environment_id
        hex_grid,
    )
    problem = HStarProblem(
        hex_grid,
        start,
        goal,
        a_max,  
        a_min,  
        start_v = start_v,
        #goal_v = goal_v,
        collision_radius = 0,
        heuristic = lambda a,b:0,#h_cost_travel_time_euclidean,
        n_step = n_step,
        
    )
    hs = BidHStarSearch(
        problem = problem,
        bidirectional = False,
        experience_collector=experience_collector,
        environment_id = seed,
    )
    fsolution = hs.search()
    f_soln_path = construct_full_solution(fsolution)


Generating maps:   0%|          | 0/20 [00:00<?, ?map/s]

In [10]:
def plot_value_network_directions(
    value_network,
    problem,
    environment,
    goal,
    velocity,
    center,
    radius,
    hex_size=1.0,
):
    value_network.eval()

    directions = (
        HexCoord( 1,  0),
        HexCoord( 1, -1),
        HexCoord( 0, -1),
        HexCoord(-1,  0),
        HexCoord(-1,  1),
        HexCoord( 0,  1),
    )

    direction_names = (
        "E",
        "SE",
        "SW",
        "W",
        "NW",
        "NE",
    )

    # Spatial context depends only on:
    # environment + goal
    #
    # So run HexUNet once.
    with torch.no_grad():
        context_map = value_network.encode_environment(
            environment,
            goal,
        )

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(18, 12),
    )

    axes = axes.flatten()

    obstacles = set(environment.obstacles)

    for ax, direction, direction_name in zip(
        axes,
        directions,
        direction_names,
    ):
        values = {}

        for q in range(
            center.q - radius,
            center.q + radius + 1,
        ):
            for r in range(
                center.r - radius,
                center.r + radius + 1,
            ):
                location = HexCoord(q, r)

                dq = q - center.q
                dr = r - center.r

                hex_distance = max(
                    abs(dq),
                    abs(dr),
                    abs(dq + dr),
                )

                if hex_distance > radius:
                    continue

                if location in obstacles:
                    continue

                state = RLState(
                    location=location,
                    velocity=VelocityState(
                        magnitude=velocity,
                        direction=direction,
                    ),
                )

                value = value_network.predict(
                    problem=problem,
                    state=state,
                    goal=goal,
                    environment=environment,
                    context_map=context_map,
                )

                values[location] = value

        plot_hex_grid_2(
            start=None,
            obstacles=obstacles,
            path=values,
            #start=problem.start,
            center=center,
            goal=goal.location,
            radius=radius,
            hex_size=hex_size,
            ax=ax,
        )

        ax.set_title(
            f"{direction_name}: {direction}"
        )

    plt.tight_layout()
    plt.show()

In [11]:
empty_environment = HexGrid(
    hex_size,
    obstacles=set(),
    center=center,
    radius=radius,
)

In [12]:
@dataclass(frozen=True)
class RelaxedTarget:
    state: RLState
    goal: RLState
    target: float


class RelaxedPretrainingDataset(Dataset):

    def __init__(self, samples):
        self.samples = list(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

In [13]:
def valid_hex_locations(
    center,
    radius,
):
    locations = []

    for q in range(
        center.q - radius,
        center.q + radius + 1,
    ):
        for r in range(
            center.r - radius,
            center.r + radius + 1,
        ):
            dq = q - center.q
            dr = r - center.r

            distance = max(
                abs(dq),
                abs(dr),
                abs(dq + dr),
            )

            if distance <= radius:
                locations.append(
                    HexCoord(q, r)
                )

    return locations

In [14]:
def generate_relaxed_pretraining_dataset(
    center,
    radius,
    goal_locations,
    velocity_values,
    acceleration,
    hex_size,
    samples_per_goal=None,
):
    locations = valid_hex_locations(
        center=center,
        radius=radius,
    )

    samples = []

    for goal_location in goal_locations:

        goal = RLState(
            location=goal_location,
            velocity=VelocityState(
                magnitude=0.0,
                direction=None,
            ),
        )

        goal_samples = []

        for location in locations:

            for velocity in velocity_values:

                state = RLState(
                    location=location,
                    velocity=VelocityState(
                        magnitude=float(velocity),
                        direction=None,
                    ),
                )

                target = travel_time_euclidean(
                    location,
                    goal_location,
                    velocity,
                    acceleration,
                    hex_size,
                )

                if not torch.isfinite(
                    torch.tensor(target)
                ):
                    continue

                goal_samples.append(
                    RelaxedTarget(
                        state=state,
                        goal=goal,
                        target=float(target),
                    )
                )

        if (
            samples_per_goal is not None
            and len(goal_samples) > samples_per_goal
        ):
            goal_samples = random.sample(
                goal_samples,
                samples_per_goal,
            )

        samples.extend(
            goal_samples
        )

    return RelaxedPretrainingDataset(
        samples
    )

In [15]:
def split_relaxed_dataset(
    dataset,
    validation_fraction=0.10,
    seed=42,
):
    indices = list(
        range(len(dataset))
    )

    generator = random.Random(seed)
    generator.shuffle(indices)

    validation_count = round(
        len(indices)
        * validation_fraction
    )

    validation_indices = indices[
        :validation_count
    ]

    training_indices = indices[
        validation_count:
    ]

    training_samples = [
        dataset.samples[index]
        for index in training_indices
    ]

    validation_samples = [
        dataset.samples[index]
        for index in validation_indices
    ]

    return (
        RelaxedPretrainingDataset(
            training_samples
        ),
        RelaxedPretrainingDataset(
            validation_samples
        ),
    )

In [16]:
def relaxed_collate(samples):
    return samples

In [17]:
directions = (
    HexCoord(1, 0),
    HexCoord(1, -1),
    HexCoord(0, -1),
    HexCoord(-1, 0),
    HexCoord(-1, 1),
    HexCoord(0, 1),
)
velocity_values = (
    2.5,
    5.0,
    7.5,
    10.0,
    12.5,
    15.0,
    17.5,
    20.0,
)

In [18]:
def relaxed_batch_loss(
    model,
    samples,
    empty_environment,
    loss_fn,
):
    if not samples:
        return None

    predictions = []
    targets = []

    samples_by_goal = defaultdict(list)

    for sample in samples:
        samples_by_goal[
            sample.goal.location
        ].append(sample)

    for _, goal_samples in samples_by_goal.items():

        goal = goal_samples[0].goal

        # One HexUNet pass per unique goal.
        context_map = model.encode_environment(
            environment=empty_environment,
            goal=goal,
        )

        for sample in goal_samples:

            prediction = model.value(
                problem=None,
                state=sample.state,
                goal=sample.goal,
                environment=empty_environment,
                context_map=context_map,
            )

            predictions.append(
                prediction
            )

            targets.append(
                sample.target
            )

    predictions = torch.stack(
        predictions
    )

    targets = torch.tensor(
        targets,
        dtype=torch.float32,
        device=predictions.device,
    )

    loss = loss_fn(
        predictions,
        targets,
    )

    return (
        loss,
        predictions,
        targets,
    )

In [19]:
def evaluate_relaxed_model(
    model,
    data_loader,
    empty_environment,
    loss_fn,
):
    model.eval()

    squared_errors = []
    absolute_errors = []

    with torch.no_grad():

        for samples in data_loader:

            result = relaxed_batch_loss(
                model=model,
                samples=samples,
                empty_environment=empty_environment,
                loss_fn=loss_fn,
            )

            if result is None:
                continue

            _, predictions, targets = result

            errors = predictions - targets

            squared_errors.append(
                errors.pow(2)
            )

            absolute_errors.append(
                errors.abs()
            )

    if not squared_errors:
        return {
            "mse": 0.0,
            "rmse": 0.0,
            "mae": 0.0,
        }

    squared_errors = torch.cat(
        squared_errors
    )

    absolute_errors = torch.cat(
        absolute_errors
    )

    mse = squared_errors.mean()

    return {
        "mse": mse.item(),
        "rmse": torch.sqrt(mse).item(),
        "mae": absolute_errors.mean().item(),
    }

In [20]:
def split_relaxed_dataset_by_goal(
    dataset,
    validation_fraction=0.2,
    seed=42,
):
    goal_locations = list({
        sample.goal.location
        for sample in dataset.samples
    })

    rng = random.Random(seed)
    rng.shuffle(goal_locations)

    validation_count = max(
        1,
        round(
            len(goal_locations)
            * validation_fraction
        ),
    )

    validation_goals = set(
        goal_locations[:validation_count]
    )

    train_samples = []
    validation_samples = []

    for sample in dataset.samples:

        if sample.goal.location in validation_goals:
            validation_samples.append(
                sample
            )
        else:
            train_samples.append(
                sample
            )

    return (
        RelaxedPretrainingDataset(
            train_samples
        ),
        RelaxedPretrainingDataset(
            validation_samples
        ),
        validation_goals,
    )

In [21]:
from tqdm.auto import tqdm


def pretrain_relaxed_value_network(
    model,
    train_loader,
    validation_loader,
    empty_environment,
    epochs=20,
    learning_rate=1e-3,
    device="cpu",
):
    model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    loss_fn = nn.MSELoss()

    history = []

    epoch_progress = tqdm(
        range(epochs),
        desc="Pretraining",
        unit="epoch",
    )

    for epoch in epoch_progress:

        model.train()

        epoch_squared_error = 0.0
        epoch_samples = 0

        batch_progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{epochs}",
            unit="batch",
            leave=False,
        )

        for samples in batch_progress:

            result = relaxed_batch_loss(
                model=model,
                samples=samples,
                empty_environment=empty_environment,
                loss_fn=loss_fn,
            )

            if result is None:
                continue

            loss, predictions, targets = result

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            n = len(targets)

            epoch_squared_error += (
                loss.item() * n
            )

            epoch_samples += n

            batch_progress.set_postfix(
                mse=f"{loss.item():.4f}",
                rmse=f"{loss.item() ** 0.5:.4f}",
            )

        train_mse = (
            epoch_squared_error
            / epoch_samples
            if epoch_samples
            else 0.0
        )

        validation_metrics = evaluate_relaxed_model(
            model=model,
            data_loader=validation_loader,
            empty_environment=empty_environment,
            loss_fn=loss_fn,
        )

        metrics = {
            "epoch": epoch,
            "train_mse": train_mse,
            "train_rmse": train_mse ** 0.5,
            "validation_mse":
                validation_metrics["mse"],
            "validation_rmse":
                validation_metrics["rmse"],
            "validation_mae":
                validation_metrics["mae"],
        }

        history.append(metrics)

        epoch_progress.set_postfix(
            train_rmse=(
                f"{metrics['train_rmse']:.4f}"
            ),
            val_rmse=(
                f"{metrics['validation_rmse']:.4f}"
            ),
            val_mae=(
                f"{metrics['validation_mae']:.4f}"
            ),
        )

    return history

In [22]:
all_locations = valid_hex_locations(
    center=center,
    radius=radius,
)

goal_locations = random.sample(
    all_locations,
    min(50, len(all_locations)),
)

dataset = generate_relaxed_pretraining_dataset(
    center=center,
    radius=radius,
    goal_locations=goal_locations,
    velocity_values=[
        0,
        2,
        5,
        10,
        15,
        20,
    ],
    acceleration=a_max,
    hex_size=hex_size,
    samples_per_goal=1000,
)


In [24]:
train_dataset, validation_dataset, validation_goals = (
    split_relaxed_dataset_by_goal(
        dataset,
        validation_fraction=0.2,
        seed=42,
    )
)

print(
    "Train samples:",
    len(train_dataset),
)

print(
    "Validation samples:",
    len(validation_dataset),
)

print(
    "Held-out goals:",
    len(validation_goals),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=relaxed_collate,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=relaxed_collate,
)

Train samples: 40000
Validation samples: 10000
Held-out goals: 10


In [ ]:

history = pretrain_relaxed_value_network(
    model=value_network,
    train_loader=train_loader,
    validation_loader=validation_loader,
    empty_environment=empty_environment,
    epochs=1,
    learning_rate=1e-3,
)

Pretraining:   0%|          | 0/1 [00:00<?, ?epoch/s]

Epoch 1/1:   0%|          | 0/625 [00:00<?, ?batch/s]

In [ ]:
history

In [ ]:
import matplotlib.pyplot as plt

epochs = [
    h["epoch"]
    for h in history
]

train_rmse = [
    h["train_rmse"]
    for h in history
]

validation_rmse = [
    h["validation_rmse"]
    for h in history
]

plt.plot(
    epochs,
    train_rmse,
    label="Train RMSE",
)

plt.plot(
    epochs,
    validation_rmse,
    label="Validation RMSE",
)

plt.xlabel("Epoch")
plt.ylabel("Travel-time RMSE")
plt.title("Relaxed Heuristic Pretraining")
plt.legend()
plt.show()

In [ ]:
s = replay_buffer.sample(100)[-22]
print(s.goal)
plot_value_network_directions(
    value_network,
    problem,
    environment_store.get(s.environment_id),
    goal=s.goal,
    velocity=6,
    center=center,
    radius=radius,
    hex_size=hex_size,
)

In [ ]:
plot_value_network_directions(
    value_network,
    problem,
    empty_environment,
    goal=s.goal,
    velocity=6,
    center=center,
    radius=radius,
    hex_size=hex_size,
)

In [ ]:
plot_hex_grid_2(
    obstacles=obstacles,
    start=goal,
    goal=goal,
    center=center,
    radius=radius,
    hex_size=hex_size,
    #path=predictions,
    #ax=ax,
    draw_axes=False,
    show_axis_legend=False,
    figsize=(24,24)
)

In [ ]:

def train_model(
    trainer,
    problem,
    search_callback=None,
    steps=10000,
    batch_size=64,
    search_every=100,
    search_when_exhausted=True,
    print_every=100,
    plot=True,
):
    losses = []

    progress = tqdm(
        range(steps),
        desc="Training",
        unit="step",
    )

    for step in progress:
    
        periodic_search = (
            search_callback is not None
            and search_every
            and step % search_every == 0
        )
    
        exhausted_search = (
            search_callback is not None
            and search_when_exhausted
            and not trainer.replay_buffer.has_training_batch(
                batch_size=batch_size,
                solution_weight=trainer.solution_weight,
                nonterminal_weight=trainer.nonterminal_weight,
            )
        )
    
        if periodic_search:
            search_callback()
        
        if (
            search_callback is not None
            and search_when_exhausted
        ):
            attempts = 0
            max_attempts = 20
            
            while (
                not trainer.replay_buffer.has_training_batch(
                    batch_size=batch_size,
                    solution_weight=trainer.solution_weight,
                    nonterminal_weight=trainer.nonterminal_weight,
                )
                and attempts < max_attempts
            ):
                search_callback()
                attempts += 1
    
        metrics = trainer.train_step(
            problem,
            batch_size=batch_size,
        )

        losses.append(metrics)

        progress.set_postfix(
            loss=f"{metrics['loss']:.4f}",
            solution_loss=f"{metrics['solution_loss']:.4f}",
            td_loss=f"{metrics['td_loss']:.4f}",
        )

        if print_every and step % print_every == 0:
            tqdm.write(
                f"{step}: {metrics} | "
                f"buffer={len(trainer.replay_buffer)}"
            )

    if plot:
        loss_values = [
            metrics["loss"]
            for metrics in losses
        ]

        plt.plot(loss_values)
        plt.xlabel("Training Step")
        plt.ylabel("MSE Loss")
        plt.title("TD Training Loss")
        plt.show()

    return losses


In [ ]:
def collect_experience():

    environment_id = random.choice(
        list(environment_store.environments.keys())
    )

    hex_grid = environment_store.get(
        environment_id
    )

    start, goal = random_start_goal(
        center,
        radius,
        min_distance_ratio=0.5,
    )

    
    problem = HStarProblem(
        hex_grid,
        start,
        goal,
        a_max,  
        a_min,  
        start_v = start_v,
        goal_v = goal_v,
        collision_radius = 0,
        heuristic = lambda a,b:0,#h_cost_travel_time_euclidean,
        n_step = n_step,   
    )
    
    hs = BidHStarSearch(
        problem=problem,
        bidirectional=False,
        experience_collector=experience_collector,
        environment_id=environment_id,
    )

    solution_path = hs.search()

    # trainer.train_solution(
    #     solution_path,
    #     hex_grid,
    #     problem.goal,
    # )

In [ ]:
# losses = train_model(
#     trainer,
#     problem,
#     search_callback=collect_experience,
#     steps=2000,
#     batch_size=16,
#     search_every=25,
#     print_every=50,
# )

In [ ]:
losses = train_model(
    trainer,
    problem,
    search_callback=collect_experience,
    steps=1000,
    batch_size=64,
    #search_every=10,
    print_every=50,
)

In [ ]:
for sample in replay_buffer.sample(3):

    environment = environment_store.get(
        sample.environment_id
    )
    
    prediction = value_network.predict(
        problem=problem,
        state=sample.state,
        goal=sample.goal,
        environment=environment,
    )

    euclidian_h = travel_time_euclidean(
        sample.state.location, 
        sample.goal.location, 
        sample.state.velocity.magnitude, 
        problem.a_min, 
        problem.grid.hex_size
    )
    if sample.terminal:
        target = sample.cost
    else:
        next_prediction = value_network.predict(
            problem=problem,
            state=sample.next_state,
            goal=sample.goal,
            environment=environment,
        )
    
        target = sample.cost + next_prediction
    print("------------------------------------------------")
    print("State:", sample.state)
    print("Goal:", sample.goal)
    print("Environment:", sample.environment_id)
    print("Euclidian h:", euclidian_h)
    
    print("Predicted h:", prediction)
    print("TD target:", target)
    print("Difference:", prediction - target)
print("------------------------------------------------")


In [ ]:

def plot_value_network_directions(
    value_network,
    problem,
    environment,
    goal,
    velocity: float,
    center,
    radius: int,
    hex_size: float = 1.0,
):
    """Plot predicted h across the map for the six primary hex directions."""

    directions = (
        HexCoord( 1,  0),  # E
        HexCoord( 1, -1),  # SE
        HexCoord( 0, -1),  # SW
        HexCoord(-1,  0),  # W
        HexCoord(-1,  1),  # NW
        HexCoord( 0,  1),  # NE
    )

    direction_names = (
        "E",
        "SE",
        "SW",
        "W",
        "NW",
        "NE",
    )

    value_network.eval()

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(18, 12),
    )

    axes = axes.flatten()

    # Run the expensive spatial network only once.
    with torch.no_grad():
        value_map = value_network.value_map(
            environment,
            goal.location,
        )

    obstacles = set(environment.obstacles)

    for ax, direction, direction_name in zip(
        axes,
        directions,
        direction_names,
    ):
        predictions = {}

        # Find the ordinal index corresponding to this primary direction.
        direction_index = problem.DIRECTIONS.index(
            direction
        )

        for q in range(-radius, radius + 1):
            for r in range(-radius, radius + 1):

                location = HexCoord(
                    center.q + q,
                    center.r + r,
                )

                if hdist(center, location) > radius:
                    continue

                if location in obstacles:
                    continue

                state = RLState(
                    location=location,
                    velocity=VelocityState(
                        magnitude=velocity,
                        direction=direction_index,
                    ),
                )

                prediction = value_network.predict(
                    problem=problem,
                    state=state,
                    goal=goal,
                    environment=environment,
                    value_map=value_map,
                )

                predictions[location] = prediction

        plot_hex_grid_2(
            obstacles=obstacles,
            start=goal.location,
            goal=goal.location,
            center=center,
            radius=radius,
            hex_size=hex_size,
            path=predictions,
            ax=ax,
            draw_axes=False,
            show_axis_legend=False,
            plt_title=(
                f"{direction_name} "
                f"{direction} | "
                f"v={velocity:.2f}"
            ),
        )

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_value_network_directions(
    value_network,
    problem,
    environment,
    goal,
    velocity,
    center,
    radius,
    hex_size=1.0,
):
    value_network.eval()

    directions = (
        HexCoord( 1,  0),
        HexCoord( 1, -1),
        HexCoord( 0, -1),
        HexCoord(-1,  0),
        HexCoord(-1,  1),
        HexCoord( 0,  1),
    )

    direction_names = (
        "E",
        "SE",
        "SW",
        "W",
        "NW",
        "NE",
    )

    # Spatial context depends only on:
    # environment + goal
    #
    # So run HexUNet once.
    with torch.no_grad():
        context_map = value_network.encode_environment(
            environment,
            goal,
        )

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(18, 12),
    )

    axes = axes.flatten()

    obstacles = set(environment.obstacles)

    for ax, direction, direction_name in zip(
        axes,
        directions,
        direction_names,
    ):
        values = {}

        for q in range(
            center.q - radius,
            center.q + radius + 1,
        ):
            for r in range(
                center.r - radius,
                center.r + radius + 1,
            ):
                location = HexCoord(q, r)

                dq = q - center.q
                dr = r - center.r

                hex_distance = max(
                    abs(dq),
                    abs(dr),
                    abs(dq + dr),
                )

                if hex_distance > radius:
                    continue

                if location in obstacles:
                    continue

                state = RLState(
                    location=location,
                    velocity=VelocityState(
                        magnitude=velocity,
                        direction=direction,
                    ),
                )

                with torch.no_grad():
                    value = value_network.value(
                        problem=problem,
                        state=state,
                        goal=goal,
                        environment=environment,
                        context_map=context_map,
                    )

                values[location] = value.item()

        plot_hex_grid_2(
            start=None,
            obstacles=obstacles,
            path=values,
            goal=goal.location,
            center=center,
            radius=radius,
            hex_size=hex_size,
            ax=ax,
        )

        ax.set_title(
            f"{direction_name}: {direction}"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
for s in replay_buffer.sample(5):
    print(s.goal)
    plot_value_network_directions(
        value_network,
        problem,
        environment_store.get(s.environment_id),
        goal=s.goal,
        velocity=6,
        center=center,
        radius=radius,
        hex_size=hex_size,
    )

In [ ]:
def get_value_network_directions(
    value_network,
    problem,
    environment,
    goal,
    location,
    velocity,
):
    """
    Return predicted h = V(s) at one hex for the six primary
    hex directions, holding location and velocity constant.
    """

    directions = {
        "E":  HexCoord( 1,  0),
        "SE": HexCoord( 1, -1),
        "SW": HexCoord( 0, -1),
        "W":  HexCoord(-1,  0),
        "NW": HexCoord(-1,  1),
        "NE": HexCoord( 0,  1),
    }

    value_network.eval()

    with torch.no_grad():

        context_map = value_network.encode_environment(
            environment,
            goal,
        )

        values = {}

        for name, direction in directions.items():

            state = RLState(
                location=location,
                velocity=VelocityState(
                    magnitude=velocity,
                    direction=direction,
                ),
            )

            value = value_network.value(
                problem=problem,
                state=state,
                goal=goal,
                environment=environment,
                context_map=context_map,
            )

            values[name] = value.item()

    return values

In [ ]:
values = get_value_network_directions(
    value_network=value_network,
    problem=problem,
    environment=environment_store.get(
        s.environment_id
    ),
    goal=goal,
    location=HexCoord(5, -4),
    velocity=2,
)

print(values)

In [ ]:
values = get_value_network_directions(
    value_network=value_network,
    problem=problem,
    environment=empty_environment,
    goal=goal,
    location=HexCoord(5, -4),
    velocity=2,
)

print(values)

In [ ]:
goal

In [ ]:
s = replay_buffer.sample(100)[-1]

goal = RLState(
    location=problem.goal,
    velocity=problem.goal_v,
)

plot_value_network_directions(
    value_network,
    problem,
    empty_environment,
    goal=goal,
    velocity=2,
    center=center,
    radius=radius,
    hex_size=hex_size,
)

In [ ]:
st = RLState(
    location=HexCoord(5, -4),
    velocity=VelocityState(
        magnitude=2,
        direction=HexCoord(1, 0),
    ),
)


print(
    "target:",
    travel_time_euclidean(
        st.location,
        goal.location,
        st.velocity.magnitude,
        a_min,
        hex_size,
    )
)

In [ ]:
def direction_sensitivity(
    value_network,
    problem,
    environment,
    goal,
    location,
    velocity,
    directions,
):
    with torch.no_grad():
        context_map = value_network.encode_environment(
            environment,
            goal,
        )

    values = []

    for direction in directions:
        state = RLState(
            location=location,
            velocity=VelocityState(
                magnitude=velocity,
                direction=direction,
            ),
        )

        value = value_network.predict(
            problem=problem,
            state=state,
            goal=goal,
            environment=environment,
            context_map=context_map,
        )

        values.append(value)

    return values

In [ ]:
samples = replay_buffer.sample_solution(10)

for sample in samples:
    print(
        sample.state.location,
        sample.state.velocity.magnitude,
        sample.state.velocity.direction,
    )

In [ ]:
sample = replay_buffer.sample_solution(1)[0]

print("state:")
print(sample.state)

print("\nencoded:")
print(
    value_network.encode_state(
        sample.state,
        sample.goal,
        environment,
    )
)

In [ ]:
location = HexCoord(-20, 9)
speed = 10.0

directions = [
    HexCoord(1, 0),
    HexCoord(1, -1),
    HexCoord(0, -1),
    HexCoord(-1, 0),
    HexCoord(-1, 1),
    HexCoord(0, 1),
]

for direction in directions:

    state = RLState(
        location=location,
        velocity=VelocityState(
            magnitude=speed,
            direction=direction,
        ),
    )

    encoded = value_network.encode_state(
        state,
        goal,
        environment,
    )

    print(
        direction,
        encoded.tolist(),
    )

In [ ]:
with torch.no_grad():
    context_map = value_network.encode_environment(
        environment,
        goal,
    )


In [ ]:
for direction in directions:

    state = RLState(
        location=location,
        velocity=VelocityState(
            magnitude=speed,
            direction=direction,
        ),
    )

    state_features = value_network.encode_state(
        state,
        goal,
        environment,
    )

    value = value_network.predict(
        problem=problem,
        state=state,
        goal=goal,
        environment=environment,
        context_map=context_map,
    )

    print(
        direction,
        state_features.tolist(),
        f"V={value:.6f}",
    )

In [ ]:
from collections import Counter

def solution_direction_counts(
    replay_buffer,
):
    samples = replay_buffer.sample_solution(
        replay_buffer.solution_size
    )

    counts = Counter()

    for sample in samples:
        direction = sample.state.velocity.direction

        key = (
            None
            if direction is None
            else (direction.q, direction.r)
        )

        counts[key] += 1

    return counts